# Read Generated Data

This notebook loads one generated `.h5` dataset file or all shards in a directory,
prints a short summary, plots histograms of the open parameters, and plots a few representative spectra.

It uses `ibamlkit.data.DatasetBatchReader` directly.

In [ ]:
from pathlib import Path
import math
import sys 
sys.path.append("../")

import matplotlib.pyplot as plt
import numpy as np

from ibamlkit.data import DatasetBatchReader
from ibamlkit.schema import IBADataset

In [ ]:
# Set this to either one .h5 file or a directory containing .h5 shards.
input_path = Path("examples/datasets/bzcy_1000")
bins = 50
n_spectra = 2

reader = DatasetBatchReader()

In [ ]:
def plot_open_parameter_histograms(dataset: IBADataset, bins: int) -> plt.Figure:
    open_parameters = list(dataset.input_spec.open_parameters)
    values = np.asarray(dataset.open_parameter_values, dtype=np.float32)
    n_params = len(open_parameters)
    ncols = min(3, max(1, n_params))
    nrows = math.ceil(n_params / ncols)
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 3.5 * nrows))
    axes_array = np.atleast_1d(axes).ravel()
    for axis, parameter, column in zip(axes_array, open_parameters, range(n_params)):
        axis.hist(values[:, column], bins=bins, color="#4472c4", edgecolor="black", alpha=0.85)
        axis.set_title(parameter.name)
        axis.set_xlabel(parameter.unit or parameter.kind)
        axis.set_ylabel("Count")
        if parameter.lower_bound is not None:
            axis.axvline(parameter.lower_bound, color="red", linestyle="--", linewidth=1)
        if parameter.upper_bound is not None:
            axis.axvline(parameter.upper_bound, color="red", linestyle="--", linewidth=1)
    for axis in axes_array[n_params:]:
        axis.set_visible(False)
    fig.suptitle("Open Parameter Histograms", fontsize=14)
    fig.tight_layout()
    return fig


def representative_indices(sample_count: int, n_spectra: int) -> list[int]:
    if sample_count <= 0 or n_spectra <= 0:
        return []
    if sample_count <= n_spectra:
        return list(range(sample_count))
    positions = np.linspace(0, sample_count - 1, num=n_spectra, dtype=int)
    return sorted(set(int(position) for position in positions))


def plot_representative_spectra(dataset: IBADataset, n_spectra: int) -> plt.Figure:
    methods = list(dataset.input_spec.methods)
    fig, axes = plt.subplots(nrows=len(methods), ncols=1, figsize=(8, 3.5 * max(1, len(methods))))
    axes_array = np.atleast_1d(axes).ravel()
    for axis, method in zip(axes_array, methods):
        spectra = np.asarray(dataset.spectra[method.name], dtype=np.float32)
        lengths = None
        if dataset.spectra_lengths is not None:
            lengths = np.asarray(dataset.spectra_lengths[method.name], dtype=np.int32)
        if spectra.shape[0] == 0:
            axis.set_title(f"{method.name}: no spectra")
            axis.set_xlabel("Channel")
            axis.set_ylabel("Counts")
            continue
        indices = representative_indices(spectra.shape[0], n_spectra)
        for index in indices:
            if lengths is not None:
                valid_length = int(lengths[index])
                x = np.arange(valid_length)
                y = spectra[index, :valid_length]
            else:
                x = np.arange(spectra.shape[1])
                y = spectra[index]
            axis.plot(x, y, label=f"sample {index}")
        axis.set_title(f"{method.name} representative spectra")
        axis.set_xlabel("Channel")
        axis.set_ylabel("Counts")
        axis.legend()
    fig.tight_layout()
    return fig

In [ ]:
paths = reader.collect_dataset_paths(input_path)
dataset = reader.load_many(paths)
reader.print_summary(dataset, paths)

In [ ]:
hist_fig = plot_open_parameter_histograms(dataset, bins=bins)
plt.show()

In [ ]:
spectra_fig = plot_representative_spectra(dataset, n_spectra=n_spectra)
plt.show()